# 03 · Zero-shot, lab → field and few-shot on PlantDoc

In [ ]:
import sys
from pathlib import Path
here = Path.cwd()
sys.path.insert(0, str(here if (here / "nbtools.py").exists() else here / "notebooks"))
from nbtools import run, show, figure, where, ROOT
where()

## Step 1 · Zero-shot text classification (both prompt styles)

In [ ]:
run("zeroshot.py")

In [ ]:
show("results/zeroshot_summary.csv")

## Step 2 · Lab → field gap, few-shot learning (K = 1, 5, 10, 16) and 5-way episodes
Expected highlight: BioCLIP 2 trained on lab images only reaches **57.6%** on PlantDoc.

In [ ]:
run("fewshot.py")

In [ ]:
show("results/fewshot/lab_to_field.csv")

In [ ]:
show("results/fewshot/fewshot_summary.csv")

## Try it yourself · the main result in 6 lines
This is the advantage of a notebook: you can poke at the data directly. The cell below loads the saved BioCLIP 2
features, trains the lab classifier and tests it on PlantDoc — it should print about **97.4% → 57.6%**.
Change `key` to another model (e.g. `"clip_l14"`, `"resnet50"`) and `C` (1000 for the two CLIP models, 100 for others).

In [ ]:
import sys, numpy as np
sys.path.insert(0, str(ROOT / "src"))
from fewshot import load_feats
from sklearn.linear_model import LogisticRegression

key, C = "bioclip2", 100
Xtr, ytr, _ = load_feats(key, "plantvillage_train_sub")
Xlab, ylab, _ = load_feats(key, "plantvillage_test_sub")
Xfield, yfield, _ = load_feats(key, "plantdoc_test")
clf = LogisticRegression(C=C, max_iter=3000).fit(Xtr, ytr)
print(f"{key}: lab {100*np.mean(clf.predict(Xlab)==ylab):.1f}%  ->  PlantDoc {100*np.mean(clf.predict(Xfield)==yfield):.1f}%")

## Optional · lab → field with all PlantVillage images (needs the optional cell in notebook 02)

In [ ]:
# run("fewshot.py", "--models", "clip_b16", "--lab", "full")